# 02. 윈도우 데이터 검증 + 라벨 적용

**목적**: 재영이 만든 이벤트 라벨과 윈도우(A, B)가 우리 규칙대로 되어 있는지 확인하고, 확정한 라벨(labels.csv)을 적용했을 때 라벨별 개수를 본다. 결과는 재영 리뷰 코멘트로 쓴다.

**준비**
1. 구글 드라이브에 `cloudtrail_data` 폴더를 만들고 거기에 넣기
    - `windows_B_size5_stride1.zip` (재영이 준 zip)
    - `labels.csv`
2. 런타임은 CPU로 충분 (GPU 필요 없음)

**칸마다 보는 법**: `[하는 일]` 이 칸이 하는 일 / `[볼 것]` 결과에서 확인할 것 / `[확인 질문]` 직접 답해보기

## 0. 드라이브 연결, 압축 풀기

[하는 일] 드라이브의 zip을 Colab 작업 폴더로 풀어둔다.

[볼 것] 파일 5개(event_labels, events_A, events_B, windows_A, windows_B)가 보이면 성공.

In [ ]:
import os, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/cloudtrail_data'
except ImportError:                      # Colab이 아닌 곳에서 돌릴 때
    DATA_DIR = os.environ.get('DATA_DIR', '.')

WORK = '/content/data' if os.path.exists('/content') else os.path.join(DATA_DIR, 'unzipped')
os.makedirs(WORK, exist_ok=True)
zips = [f for f in os.listdir(DATA_DIR) if f.endswith('.zip')]
for z in zips:
    zipfile.ZipFile(os.path.join(DATA_DIR, z)).extractall(WORK)
print('zip:', zips)
print('풀린 파일:', sorted(f for f in os.listdir(WORK) if f.endswith('.csv')))

## 1. 불러오기와 전체 모습

[하는 일] 이벤트 파일과 윈도우 파일을 읽고 크기를 본다.

[볼 것] 이벤트 약 2.2만 개, 윈도우 A 약 2.2만 개 / B 약 4.7천 개.

In [ ]:
import pandas as pd, numpy as np
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 120)

ev = pd.read_csv(f'{WORK}/event_labels.csv', low_memory=False)
events = {k: pd.read_csv(f'{WORK}/events_{k}.csv', low_memory=False) for k in 'AB'}
wins = {k: pd.read_csv(f'{WORK}/windows_{k}_size5_stride1.csv', low_memory=False) for k in 'AB'}

print(f'이벤트 {len(ev):,}개 | 공격 {ev.is_attack.sum():,} / 정상 {(~ev.is_attack).sum():,} | 출처 {ev.source.value_counts().to_dict()}')
for k in 'AB':
    print(f'[{k}] 압축 후 이벤트 {len(events[k]):,}개 | 윈도우 {len(wins[k]):,}개')
print('\n윈도우 파일 칸:', list(wins['A'].columns))

## 2. 세션 구성 확인

[하는 일] 세션(시나리오)마다 공격 이벤트 비율을 본다.

[볼 것] 순수 공격 / 순수 정상 / 섞인 세션 수, 정상 이벤트가 어느 세션에서 나오는지.

[확인 질문] 정상 이벤트가 몇 개 세션에서 나오나? 이게 왜 약점일까?

In [ ]:
s = ev.groupby('session').is_attack.agg(['mean', 'size'])
print('세션', len(s), '| 순수 공격', (s['mean'] == 1).sum(), '| 순수 정상', (s['mean'] == 0).sum(),
      '| 섞임', ((s['mean'] > 0) & (s['mean'] < 1)).sum())
print('\n정상 이벤트가 나온 세션:')
print(ev[~ev.is_attack].session.value_counts())

## 3. 방식 1 정답 규칙이 맞게 들어갔나

[하는 일] 윈도우에 들어 있는 이벤트들의 라벨로 정답을 **직접 다시 계산**해서, 재영 파일의 `m1_*` 칸과 비교한다.

- 방식 1: 공격 이벤트가 하나라도 있으면 공격, 그 이벤트들의 전술·기법을 전부 정답으로. 전부 정상이면 정상.

[볼 것] 불일치 0개면 규칙이 정확히 적용된 것.

[확인 질문] 불일치가 있다면 어떤 윈도우인가?

In [ ]:
def check_rule(k):
    e = events[k].set_index('event_uid')
    w = wins[k]
    bad = []
    for i, row in w.iterrows():
        uids = row.event_uids.split('|')
        sub = e.loc[uids]
        att = sub[sub.is_attack.astype(bool)]
        is_att = len(att) > 0
        tac = sorted({t for x in att.tactics.dropna() for t in x.split('|')})
        tech = sorted(set(att.technique.dropna()))
        w_tac = sorted(str(row.m1_tactics).split('|')) if is_att else []
        w_tech = sorted(str(row.m1_techniques).split('|')) if is_att else []
        if bool(row.m1_is_attack) != is_att or (is_att and (tac != w_tac or tech != w_tech)):
            bad.append(i)
    return bad

for k in 'AB':
    bad = check_rule(k)
    print(f'[{k}] 윈도우 {len(wins[k]):,}개 중 방식1 불일치 {len(bad)}개')
    if bad:
        print(wins[k].loc[bad[:3], ['session', 'window', 'm1_is_attack', 'm1_tactics', 'm1_techniques']])

## 4. 같은 문장인데 정답이 다른 윈도우 (label_conflict)

[하는 일] 재영 파일의 `label_conflict`가 무엇을 뜻하는지 확인한다. 같은 문장(signature)이 여러 세션에 나올 때, 정답이 **전술 단위에서도** 다른지, **기법만** 다른지 나눠 본다.

[볼 것] 전술은 같고 기법만 다른 경우(예: T1580 vs T1526, 둘 다 Discovery)는 우리처럼 전술 단위로 하면 충돌이 아니다. 전술까지 다른 경우만 진짜 충돌이라 뺀다.

[확인 질문] 진짜 충돌(전술이 다른 경우)은 몇 개인가?

In [ ]:
for k in 'AB':
    w = wins[k]
    c = w[w.label_conflict.astype(bool)].copy()
    c['tac'] = np.where(c.m1_is_attack.astype(bool), c.m1_tactics.fillna(''), 'BENIGN')
    by_sig = c.groupby('signature').agg(tac=('tac', lambda x: tuple(sorted(set(x)))),
                                        tech=('m1_techniques', lambda x: tuple(sorted(set(x.fillna(''))))))
    real = by_sig[by_sig.tac.apply(len) > 1]
    print(f'[{k}] 충돌 표시 윈도우 {len(c):,}개 = 문장 {len(by_sig)}종류')
    print(f'     기법만 다름(전술은 같음) {len(by_sig) - len(real)}종류 / 전술까지 다름 {len(real)}종류')
    print('     전술까지 다른 조합:', real.tac.value_counts().to_dict())

## 5. 정답 누수 확인

[하는 일] 윈도우 문장 재료(signature)에 정답을 알려주는 정보가 들어갔는지 본다.

- userAgent의 도구 이름 (stratus, pacu 등), IP 주소, 시나리오 이름이 있으면 안 된다.

[볼 것] signature 한 칸이 어떤 필드로 이뤄졌는지, 금지어가 0개인지.

[확인 질문] signature에 들어간 필드 중 빼야 할 게 있나?

In [ ]:
sig_example = wins['A'].signature.iloc[0].split(' > ')[0]
print('이벤트 하나의 signature 예:', sig_example)
print('필드 순서 (추정): eventSource | eventName | errorCode | readOnly | userType | region_changed | repeat_count | actor')

import re
banned = ['stratus', 'pacu', 'atomic', 'splunk', 'attack', 'redteam']
ip_re = re.compile(r'\b\d{1,3}(?:\.\d{1,3}){3}\b')
for k in 'AB':
    sig = wins[k].signature.fillna('').str.lower()
    hits = {b: int(sig.str.contains(b).sum()) for b in banned}
    print(f'[{k}] 금지어 포함 윈도우 수 {hits} | IP 포함 {int(sig.str.contains(ip_re).sum())}')

## 6. A와 B는 뭐가 다른가 (예시)

[하는 일] 같은 세션의 첫 윈도우를 A와 B로 나란히 본다.

[볼 것] B는 반복 이벤트가 압축돼서(repeat_count) 한 윈도우에 더 긴 흐름이 들어간다.

[확인 질문] 어느 쪽이 공격 흐름을 더 잘 보여주나?

In [ ]:
sess = 'stratus:aws.execution.ssm-send-command'   # 반복이 많은 세션 (B에서 압축 효과가 큼)
for k in 'AB':
    w = wins[k][wins[k].session == sess].sort_values('window')
    print(f'[{k}] 윈도우 {len(w)}개, 공격이 처음 나오는 윈도우:')
    first = w[w.m1_is_attack.astype(bool)].iloc[0] if w.m1_is_attack.astype(bool).any() else w.iloc[0]
    for e in first.signature.split(' > '):
        f = e.split('|')
        print('   ', f[0].split('.')[0], f[1], '반복', f[6])

## 7. 확정 라벨 적용

[하는 일] labels.csv의 `source_tactics`로 원래 전술을 우리 라벨(`label_id`)로 바꾸고, 중복·충돌·제외 윈도우를 뺀 뒤 라벨별 개수를 센다.

- 중복 제거: 같은 문장(signature)은 하나만 남김 (세션이 달라도)
- 충돌 제거: 같은 문장인데 **우리 라벨**이 다르면 전부 뺌
- 제외 라벨(excluded)이 붙은 윈도우는 삭제

[볼 것] 학습 라벨이 라벨당 26개(학습 16 + 검증·시험 약 10) 이상인지.

[확인 질문] k=16이 안 되는 라벨은 어떤 것인가?

In [ ]:
labels = pd.read_csv(f'{DATA_DIR}/labels.csv') if os.path.exists(f'{DATA_DIR}/labels.csv') else pd.read_csv('labels.csv')
to_label = {t: r.label_id for r in labels.itertuples() for t in str(r.source_tactics).split('|') if t and t != 'nan'}
role = dict(zip(labels.label_id, labels.role))

def apply_labels(k):
    w = wins[k]
    u = w.copy()
    tac = u.m1_tactics.where(u.m1_is_attack.astype(bool), '').fillna('')
    u['labels'] = tac.apply(lambda s: sorted({to_label[t] for t in s.split('|') if t}) or ['BENIGN'])
    u = u[~u['labels'].apply(lambda ls: any(role[l] == 'excluded' for l in ls))]
    # 같은 문장은 하나만 남긴다. 같은 문장인데 우리 라벨이 다르면 전부 뺀다
    u['key'] = u['labels'].apply('|'.join)
    n_lab = u.groupby('signature').key.nunique()
    conflict = n_lab[n_lab > 1].index
    u = u[~u.signature.isin(conflict)].drop_duplicates('signature')
    print(f'[{k}] 우리 라벨 기준 충돌 문장 {len(conflict)}종류 제거 -> 고유 윈도우 {len(u):,}개')
    return u

final = {k: apply_labels(k) for k in 'AB'}
rows = []
for k in 'AB':
    ex = final[k].explode('labels')
    cnt = ex['labels'].value_counts(); ses = ex.groupby('labels').session.nunique()
    for l in labels[labels.role != 'excluded'].label_id:
        rows.append({'data': k, 'label_id': l, 'role': role[l], 'windows': int(cnt.get(l, 0)), 'sessions': int(ses.get(l, 0))})
table = pd.DataFrame(rows).pivot_table(index=['label_id', 'role'], columns='data', values=['windows', 'sessions']).astype(int)
table['k16_A'] = np.where(table[('windows', 'A')] >= 26, 'O', 'X')
table['k16_B'] = np.where(table[('windows', 'B')] >= 26, 'O', 'X')
print({k: len(final[k]) for k in 'AB'}, '개 윈도우 사용')
table.sort_values(('windows', 'A'), ascending=False)

## 8. 재영 리뷰 코멘트 정리

위 결과를 보고 아래 표를 채워서 PR 코멘트로 남긴다.

| 항목 | 결과 | 코멘트 |
|---|---|---|
| 방식1 규칙 (3번) | 불일치 _개 | |
| 정답 충돌 (4번) | A _개 / B _개 | 학습에서 제외 |
| 누수 (5번) | 금지어 _개, IP _개 | |
| 정상 세션 (2번) | 세션 _개 | 정상 데이터 다양성 부족 |
| 라벨별 개수 (7번) | k=16 불가 라벨: _ | |